# 03 - 转换与导出

本 Notebook 展示返回渲染文本和显式写盘的区别。目标格式只能表达源计算模型的一部分；这里用 XYZ 演示坐标导出。

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory

from molop import AutoParser, molopconfig

molopconfig.quiet()
sample_candidates = (
    Path("docs/assets/examples/water_mp2.out"),
    Path("../../assets/examples/water_mp2.out"),
    Path("water_mp2.out"),
)
sample_path = next((path for path in sample_candidates if path.is_file()), None)
if sample_path is None:
    raise FileNotFoundError("请将 water_mp2.out 放在 Notebook 同目录，或使用文档中的样例文件。")
batch = AutoParser(sample_path, n_jobs=1)
rendered = batch.format_transform("xyz", frame=-1, write_to_disk=False)
print(rendered[str(sample_path.resolve())])

3
comment charge 0 multiplicity 1
O               1.7849140000      1.2624220000      0.5119850000
H               2.6482370000      1.0729290000      0.1316310000
H               1.1831680000      1.2568160000     -0.2388350000


Python API 对 batch 调用时返回从源路径到渲染内容的映射。预览调用不会创建输出文件。

In [2]:
with TemporaryDirectory() as output_dir:
    batch.format_transform(
        "xyz",
        output_dir=output_dir,
        frame=-1,
        write_to_disk=True,
    )
    print(sorted(path.name for path in Path(output_dir).iterdir()))

['water_mp2.xyz']


Gaussian 和 ORCA input writer 接收格式专用参数。提交生成的输入前，应复核 route、资源、charge、multiplicity 和溶剂设置。

In [3]:
gjf = batch[0][-1].format_transform(
    "gjf",
    route_section="#p B3LYP/6-31G(d) opt",
)
print(gjf.splitlines()[0])
print(next(line for line in gjf.splitlines() if line.startswith("#p")))

%pal
#p B3LYP/6-31G(d) opt
